In [ ]:
# In[1]:

import pandas as pd

# Load the metric_container.csv file
metric_container_path = 'dataset/Bank/telemetry/2021_03_06/metric/metric_container.csv'
metric_container_df = pd.read_csv(metric_container_path)

# Extract unique KPI names and their corresponding components
kpi_component_mapping = metric_container_df[['kpi_name', 'cmdb_id']].drop_duplicates()

kpi_component_mapping

```
Out[1]:
```
The `metric_container.csv` file contains 1,701 unique KPI names associated with various components. These KPIs include metrics related to CPU usage, memory usage, network activity, process counts, system checks, and file sizes, among others. Components such as `Tomcat04` and `apache01` are linked to these KPIs. For a complete understanding of all KPI names and their corresponding components, further exploration of the dataset is recommended.

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name   cmdb_id
0                             OSLinux-CPU_CPU_CPUCpuUtil  Tomcat04
1                 OSLinux-OSLinux_MEMORY_MEMORY_CacheMem  apache01
2               OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem  apache01
3           OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc  apache01
4                  OSLinux-OSLinux_MEMORY_MEMORY_UserMem  apache01
...                                                  ...       ...
279679  OSLinux-OSLinux_PROCESS_apache_10001_PROCPPCount  apache01
279754      OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  apache01
280847       OSLinux-NTP_197.30.1.67_NtpServerTimeOffset  apache01
286375      OSLinux-OSLinux_SYSTEM_SYSTEM_Check-Hostname  apache01
286737       OSLinux-OSLinux_FILE_-home-zabbix_DirSizeMB  apache01

[1701 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

# Filter the metric_container_df for possible root cause components
possible_root_cause_components = [
    'apache01', 'apache02', 'Tomcat01', 'Tomcat02', 'Tomcat04', 'Tomcat03',
    'MG01', 'MG02', 'IG01', 'IG02', 'Mysql01', 'Mysql02', 'Redis01', 'Redis02'
]
filtered_metric_container_df = metric_container_df[metric_container_df['cmdb_id'].isin(possible_root_cause_components)]

# Group by 'cmdb_id' and 'kpi_name' to calculate the P95 threshold for each component-KPI pair
p95_thresholds = (
    filtered_metric_container_df
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'p95_threshold'})
)

p95_thresholds

```
Out[2]:
```
The global P95 thresholds for each 'component-KPI' time series have been successfully calculated using the entire dataset. There are 1,651 unique 'component-KPI' pairs across the possible root cause components, such as `IG01`, `apache02`, and others. These thresholds represent the 95th percentile value for each KPI, which can be used to identify anomalies during the failure period. For a detailed view of specific thresholds, further exploration of the dataset is recommended.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                          kpi_name  p95_threshold
0         IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryMax   2.040136e+09
1         IG01        JVM-Memory_7778_JVM_Memory_HeapMemoryUsage   4.171149e+01
2         IG01         JVM-Memory_7778_JVM_Memory_HeapMemoryUsed   8.491976e+08
3         IG01       JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed   5.801173e+07
4         IG01         JVM-Operating System_7778_JVM_JVM_CPULoad   5.350500e-01
...        ...                                               ...            ...
1646  apache02                      OSLinux-OSLinux_SWAP_SWAP_Si   0.000000e+00
1647  apache02                      OSLinux-OSLinux_SWAP_SWAP_So   0.000000e+00
1648  apache02  OSLinux-OSLinux_SYSTEM_SYSTEM_Check-DefaultRoute   1.000000e+00
1649  apache02      OSLinux-OSLinux_SYSTEM_SYSTEM_Check-Hostname   1.000000e+00
1650  apache02                OSLinux-OSLinux_ZABBIX_Host_Uptime   6.240685e+06

[1651 rows x 3 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pytz
from datetime import datetime

# Define the time range in UTC+8
timezone = pytz.timezone('Asia/Shanghai')
start_time = timezone.localize(datetime(2021, 3, 6, 0, 30))
end_time = timezone.localize(datetime(2021, 3, 6, 1, 0))

# Convert the time range to UNIX timestamps (seconds)
start_timestamp = int(start_time.timestamp())
end_timestamp = int(end_time.timestamp())

# Filter the data within the specified time range
filtered_time_range_df = filtered_metric_container_df[
    (filtered_metric_container_df['timestamp'] >= start_timestamp) &
    (filtered_metric_container_df['timestamp'] < end_timestamp)
]

filtered_time_range_df

```
Out[3]:
```
The data within the time range of March 6, 2021, from 00:30 to 01:00 (UTC+8) has been successfully filtered. A total of 25,381 data points across various 'component-KPI' time series are retained for further analysis. These data points include metrics from components such as `IG02`, `Tomcat02`, `MG02`, and `Redis01`, among others. For a detailed view of specific data points, further exploration of the dataset is recommended.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp   cmdb_id                                           kpi_name        value
25482  1614961800      IG02  OSLinux-OSLinux_FILE_-tmp-zabbix_agentd.log_Fi...     1.065800
25483  1614961800      IG02                    OSLinux-CPU_CPU-0_SingleCpuidle    99.102095
25484  1614961800  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     0.000000
25485  1614961800  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...   500.000000
25486  1614961800      IG02  OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSCapacity     0.426812
...           ...       ...                                                ...          ...
50959  1614963540      MG02       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc    98.000000
50960  1614963540      MG02           OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem   125.000000
50961  1614963540      MG02             OSLinux-OSLinux_MEMORY_MEMORY_CacheMem  3654.000000
50962  1614963540   Redis01                         OSLinux-CPU_CPU_CPUSysTime     0.477900
50963  1614963540      MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps     0.000000

[25381 rows x 4 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

# Merge the filtered dataset with the P95 thresholds to perform anomaly detection
anomalies_df = filtered_time_range_df.merge(
    p95_thresholds,
    on=['cmdb_id', 'kpi_name'],
    how='left'
)

# Retain only the data points that exceed the P95 thresholds
anomalies_df = anomalies_df[anomalies_df['value'] > anomalies_df['p95_threshold']]

anomalies_df

```
Out[4]:
```
Anomaly detection has been successfully performed on the filtered dataset. A total of 401 data points across various 'component-KPI' time series exceed their respective global P95 thresholds, indicating potential anomalies. These anomalies span components such as `IG02`, `Mysql01`, `Tomcat02`, `Redis01`, and others. For a detailed view of specific anomalies, further exploration of the dataset is recommended.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp   cmdb_id                                           kpi_name         value  p95_threshold
180    1614961800      IG02  OSLinux-OSLinux_FILESYSTEM_-tomcat_FSAvailable...  4.242500e+04   4.240135e+04
232    1614961800   Mysql01                     Mysql-MySQL_3306_Handler Write  2.355930e+01   2.328810e+01
246    1614961800   Mysql01                        Mysql-MySQL_3306_Com Insert  2.712000e-01   0.000000e+00
258    1614961800   Mysql01     Mysql-MySQL_3306_Innodb buffer pool pages free  2.524540e+05   2.524530e+05
281    1614961800   Mysql01         Mysql-MySQL_3306_Innodb log write requests  2.355900e+00   2.333300e+00
...           ...       ...                                                ...           ...            ...
25196  1614963540   Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...  4.000000e-01   0.000000e+00
25233  1614963540  Tomcat02           OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem  6.170000e+02   5.820000e+02
25288  1614963540   Redis01          redis-Redis_6379_Redis  (used_memory_rss)  1.236582e+07   1.235722e+07
25305  1614963540  Tomcat01  OSLinux-OSLinux_FILESYSTEM_-tomcat_FSAvailable...  3.411900e+04   3.410100e+04
25339  1614963540   Redis01                            OSLinux-CPU_CPU_CPULoad  1.340000e+00   1.270000e+00

[401 rows x 5 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

# Sort anomalies by component, KPI, and timestamp
anomalies_df = anomalies_df.sort_values(by=['cmdb_id', 'kpi_name', 'timestamp'])

# Group consecutive anomalies for each 'component-KPI' time series
anomalies_df['is_consecutive'] = (
    (anomalies_df['timestamp'].diff() > 60) |  # Check if the time difference is greater than 60 seconds
    (anomalies_df['cmdb_id'] != anomalies_df['cmdb_id'].shift()) |
    (anomalies_df['kpi_name'] != anomalies_df['kpi_name'].shift())
).cumsum()

# Group by 'cmdb_id', 'kpi_name', and 'is_consecutive' to identify faults
faults = anomalies_df.groupby(['cmdb_id', 'kpi_name', 'is_consecutive']).agg(
    start_time=('timestamp', 'min'),
    end_time=('timestamp', 'max'),
    max_value=('value', 'max'),
    threshold=('p95_threshold', 'first')
).reset_index()

# Calculate the threshold breach percentage
faults['threshold_breach'] = (faults['max_value'] - faults['threshold']) / faults['threshold']

# Filter out faults where the threshold breach is <= 50%
significant_faults = faults[faults['threshold_breach'] > 0.5]

significant_faults

```
Out[5]:
```
Significant faults have been identified by grouping consecutive anomalies for each 'component-KPI' time series and filtering out noise spikes and minor threshold breaches. A total of 336 significant faults were detected across various components, such as `IG01`, `Mysql01`, `Tomcat01`, and others. These faults represent instances where the maximum value in the anomaly sub-series exceeded the threshold by more than 50%, indicating potential issues. For a detailed view of specific faults, further analysis of the dataset is recommended.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  is_consecutive  start_time    end_time     max_value     threshold  threshold_breach
19       IG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps              20  1614963300  1614963300  5.000000e+00  2.000000e+00          1.500000
21       IG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...              22  1614962700  1614962700  4.000000e-01  2.000000e-01          1.000000
25       IG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps              26  1614962760  1614962760  1.000000e+00  0.000000e+00               inf
50       IG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...              51  1614961860  1614961860  5.000000e-01  3.000000e-01          0.666667
51       IG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...              52  1614962760  1614962760  6.000000e-01  3.000000e-01          1.000000
83       MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...              84  1614961860  1614961860  9.400000e+02  2.350000e+01         39.000000
84       MG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps              85  1614961860  1614961860  8.000000e+00  1.000000e+00          7.000000
95       MG02                             OSLinux-CPU_CPU_CPUWio              96  1614963300  1614963300  1.571200e+00  8.420750e-01          0.865867
106      MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps             107  1614962040  1614962040  4.450000e+00  7.699800e-01          4.779371
107      MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead             108  1614962040  1614962040  5.030667e+02  1.584002e+01         30.759221
108      MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...             109  1614962160  1614962160  5.160000e+02  2.515000e+01         19.516899
109      MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps             110  1614962160  1614962160  5.000000e+00  1.000000e+00          4.000000
110      MG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             111  1614962220  1614962220  1.000000e+00  0.000000e+00               inf
111   Mysql01                        Mysql-MySQL_3306_Com Insert             112  1614961800  1614961800  2.712000e-01  0.000000e+00               inf
112   Mysql01                        Mysql-MySQL_3306_Com Insert             113  1614963300  1614963300  1.333000e-01  0.000000e+00               inf
124   Mysql01              Mysql-MySQL_3306_Innodb pages created             125  1614963300  1614963300  1.670000e-02  0.000000e+00               inf
129   Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps             130  1614962040  1614962040  1.116700e+00  5.500000e-01          1.030364
130   Mysql01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead             131  1614962040  1614962040  2.186670e+01  9.000025e+00          1.429627
131   Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...             132  1614962160  1614962160  3.400000e+01  2.200000e+01          0.545455
134   Mysql01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps             135  1614961980  1614961980  6.000000e+00  0.000000e+00               inf
139   Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKPer...             140  1614963000  1614963000  3.720000e+01  1.070000e+01          2.476636
140   Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKPer...             141  1614963120  1614963120  2.440000e+01  1.070000e+01          1.280374
143   Mysql02  Mysql-MySQL_3306_Innodb buffer pool pages flushed             144  1614962340  1614962340  4.832790e+01  2.816970e+01          0.715599
149   Mysql02               Mysql-MySQL_3306_Innodb data written             150  1614962340  1614962340  1.648380e+06  1.050168e+06          0.569634
152   Mysql02        Mysql-MySQL_3306_Innodb dblwr pages written             153  1614962340  1614962340  4.832790e+01  2.816970e+01          0.715599
161   Mysql02              Mysql-MySQL_3306_Innodb pages written             162  1614962340  1614962340  4.832790e+01  2.818807e+01          0.714480
172   Mysql02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps             173  1614963480  1614963480  9.000000e+00  4.700000e+00          0.914894
173   Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...             174  1614962820  1614962820  3.680000e+01  2.600000e-01        140.538462
174   Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...             175  1614962940  1614962940  1.440000e+01  2.600000e-01         54.384615
175   Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...             176  1614963360  1614963360  8.000000e-01  2.600000e-01          2.076923
176   Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps             177  1614963000  1614963000  7.766667e+00  8.550001e-01          8.083820
177   Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead             178  1614963000  1614963000  1.680000e+02  2.052668e+01          7.184471
178   Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...             179  1614963060  1614963060  1.860000e+02  3.200000e+01          4.812500
179   Mysql02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps             180  1614963060  1614963060  8.000000e+00  1.000000e+00          7.000000
181   Mysql02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite             182  1614963000  1614963000  1.880000e+01  1.222000e+01          0.538462
182   Mysql02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps             183  1614963420  1614963420  6.000000e+00  0.000000e+00               inf
183   Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...             184  1614963540  1614963540  4.000000e-01  0.000000e+00               inf
198   Redis01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps             199  1614963000  1614963000  8.000000e+00  5.000000e+00          0.600000
199   Redis01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps             200  1614963360  1614963360  1.300000e+01  5.000000e+00          1.600000
213   Redis02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps             214  1614963360  1614963360  4.000000e+00  2.000000e+00          1.000000
231  Tomcat01                             OSLinux-CPU_CPU_CPUWio             232  1614961800  1614961800  1.018500e+00  4.452100e-01          1.287684
233  Tomcat01                             OSLinux-CPU_CPU_CPUWio             234  1614963300  1614963300  2.178100e+00  4.452100e-01          3.892298
237  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead             238  1614963300  1614963300  1.386670e+01  6.000000e+00          1.311117
239  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...             240  1614963300  1614963300  5.000000e-01  3.000000e-01          0.666667
240  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps             241  1614963300  1614963300  1.893330e+01  0.000000e+00               inf
241  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead             242  1614963300  1614963300  2.690507e+04  0.000000e+00               inf
242  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...             243  1614963300  1614963300  2.788000e+04  2.390000e+02        115.652720
243  Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps             244  1614963300  1614963300  2.000000e+01  1.000000e+00         19.000000
245  Tomcat01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite             246  1614963300  1614963300  9.750667e+02  2.386667e+02          3.085474
253  Tomcat02                             OSLinux-CPU_CPU_CPUWio             254  1614963300  1614963300  2.298300e+00  5.449300e-01          3.217606
266  Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead             267  1614963300  1614963300  1.333330e+01  6.000000e+00          1.222217
269  Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps             270  1614963300  1614963300  2.016700e+00  0.000000e+00               inf
270  Tomcat03    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead             271  1614963300  1614963300  2.589200e+03  0.000000e+00               inf
276  Tomcat04                    OSLinux-CPU_CPU-2_SingleCpuUtil             277  1614963300  1614963300  1.326460e+01  5.796100e+00          1.288539
278  Tomcat04                             OSLinux-CPU_CPU_CPUWio             279  1614963300  1614963300  1.001700e+00  5.289300e-01          0.893823
285  Tomcat04       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             286  1614963180  1614963180  1.000000e+00  0.000000e+00               inf
286  Tomcat04       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             287  1614963360  1614963420  2.000000e+00  0.000000e+00               inf
322  apache02                            OSLinux-CPU_CPU_CPULoad             323  1614962340  1614962400  1.800000e-01  1.000000e-01          0.800000
324  apache02                        OSLinux-CPU_CPU_CPUUserTime             325  1614962400  1614962400  6.104000e-01  2.432288e-01          1.509571
335  apache02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps             336  1614962580  1614962580  4.000000e+00  1.000000e+00          3.000000```
```

In [ ]:
# In[6]:

# Identify the fault with the most significant deviation from the threshold
most_significant_fault = significant_faults.loc[significant_faults['threshold_breach'].idxmax()]

# Determine the level of the component associated with this fault
# Assuming the component level can be inferred from the cmdb_id format:
# - If cmdb_id starts with "Tomcat", "apache", "Redis", or "Mysql", it is a container.
# - If cmdb_id starts with "MG" or "IG", it is a node.
# - Otherwise, it is a service.

cmdb_id = most_significant_fault['cmdb_id']
if cmdb_id.startswith(('Tomcat', 'apache', 'Redis', 'Mysql')):
    component_level = 'container'
elif cmdb_id.startswith(('MG', 'IG')):
    component_level = 'node'
else:
    component_level = 'service'

most_significant_fault, component_level

```
Out[6]:
```
The fault with the most significant deviation from the threshold is associated with the component `IG01` and the KPI `OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps`. This fault occurred at the node level, with a threshold breach value of infinity, indicating an extreme deviation from the threshold.

The original code execution output of IPython Kernel is also provided below for reference:

(cmdb_id                                                       IG01
kpi_name            OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps
is_consecutive                                                  26
start_time                                              1614962760
end_time                                                1614962760
max_value                                                      1.0
threshold                                                      0.0
threshold_breach                                               inf
Name: 25, dtype: object, 'node')```
```